# 01 · FairRent Akure: data preparation and dataset

**Goal:** turn the Akure housing survey into (1) clean, anonymised data and (2) a conversation dataset for fine-tuning.

**Runs on:** CPU (no GPU needed), about 1 minute.

Steps:
1. Load the anonymised survey (the raw export with phone numbers and emails is **not** in this repo).
2. Explore what tenants reported.
3. Build typical-rent references used by the fee engine.
4. Build the conversation dataset (train/test) for Gemma fine-tuning.

In [ ]:
# --- Setup: get the project code (works in Colab and locally) ---
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/fairrent-akure.git"  # <-- change to your repo

import os, sys
if os.path.exists("../src/fairrent"):          # running locally from notebooks/
    os.chdir("..")
elif not os.path.exists("src/fairrent"):       # running in Colab
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(folder):
        os.system(f"git clone -q {REPO_URL}")
    os.chdir(folder)
sys.path.insert(0, "src")
print("Working in:", os.getcwd())

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from fairrent.engine import FeeEngine, summary_stats

# Optional: re-clean from the raw Google Form export (keep this file private).
RAW_PATH = "data/raw/survey_raw.csv"
if os.path.exists(RAW_PATH):
    !python scripts/prepare_data.py {RAW_PATH}

tenants = pd.read_csv("data/tenants_clean.csv", keep_default_na=False, na_values=[""])
landlords = pd.read_csv("data/landlords_clean.csv", keep_default_na=False, na_values=[""])
print(f"{len(tenants)} tenants, {len(landlords)} landlords")
tenants.head()

## 1. What tenants told us

Rows whose numbers do not add up (for example, total paid lower than the rent) are kept but flagged in `quality_flags`.
They are left out of rent statistics only when the rent itself is implausible.

In [ ]:
summary = summary_stats(tenants)
for k, v in summary.items():
    print(f"{k:28s} {v}")
print()
print("Rows with quality flags:", (tenants["quality_flags"].fillna("") != "").sum())
tenants["quality_flags"].replace("", pd.NA).dropna().value_counts()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

tenants["area"].value_counts().plot.barh(ax=axes[0], color="#4C72B0")
axes[0].set_title("Responses by area"); axes[0].invert_yaxis()

sc = tenants[(tenants.house_type == "Self-contain") & (tenants.rent >= 20000)]
axes[1].hist(sc["rent"] / 1000, bins=12, color="#4C72B0")
axes[1].set_title("Self-contain yearly rent (₦ thousands)")

valid = tenants[tenants.rent >= 20000]
ratios = {}
for col, label in [("agency_fee", "Agency"), ("agreement_fee", "Agreement"), ("caution_fee", "Caution")]:
    s = valid[(valid[col] > 0) & (valid[col] <= valid.rent)]
    ratios[label] = (s[col] / s.rent * 100).tolist()
axes[2].boxplot(list(ratios.values()))
axes[2].set_xticks(range(1, len(ratios) + 1), list(ratios.keys()))
axes[2].set_title("Fees as % of rent (tenants who paid them)")
plt.tight_layout(); plt.show()

In [ ]:
# Fee quartiles that set the engine's thresholds (see src/fairrent/engine.py)
for label, r in ratios.items():
    s = pd.Series(r)
    print(f"{label:10s} n={len(s):2d}  median={s.median():5.1f}%  upper quartile={s.quantile(.75):5.1f}%")

## 2. Typical rents (the engine's reference data)

The engine compares a rent with the closest group that has at least 5 reports:
house type + area first, then house type across Akure.

In [ ]:
print(json.dumps(json.load(open("data/rent_reference.json")), indent=2))

## 3. Try the fee engine

Every number FairRent shows comes from this engine, not from the language model.

In [ ]:
from fairrent.composer import compose_reply

engine = FeeEngine(tenants)
example = {"intent": "fee_check", "area": "FUTA South Gate", "house_type": "Self-contain",
           "fees": {"rent": 280000, "agency": 56000, "caution": 50000, "inspection": 5000,
                    "other": [{"name": "Development fee", "amount": 40000}]},
           "language": "pcm"}
print(compose_reply(engine.check(example)))

## 4. Build the conversation dataset

Respondents are split 70/30 into train and test **before** any messages are written,
and the test set uses its own message templates, so the model is never tested on phrasings it trained on.

Each conversation gives two training examples: **extract** (message → JSON) and **reply** (engine result → answer).

In [ ]:
!python scripts/build_dataset.py

test = [json.loads(l) for l in open("data/conversations_test.jsonl", encoding="utf-8")]
c = test[1]
print("MESSAGE:\n", c["message"])
print("\nGOLD EXTRACTION:\n", json.dumps(c["extraction"], ensure_ascii=False))
print("\nGOLD REPLY:\n", c["reply"])

In [ ]:
# Unit tests for cleaning, engine, composer and metrics
!python -m pytest -q